In [0]:
cus_df = spark.read.table("01_bronze_catalog.raw_schema.customer")

In [0]:
# dropping rows where customer_id null and filling with unknown for string columns
cus_df = cus_df.fillna("unknown",subset=["customer_name","country","industry_type"])
cus_df = cus_df.withColumnRenamed("country","country_code")

In [0]:
from pyspark.sql.functions import col
cus_df = cus_df.withColumn("is_active",col("is_active").cast("boolean")) # changing integer is_active to boolean

In [0]:
emp_df = spark.read.table("01_bronze_catalog.raw_schema.employee")

In [0]:
# filling "unknown" for string columns
emp_df = emp_df.fillna("unknown",subset=["employee_name","role","region"])

In [0]:
# changing is_active_flag to boolean
from pyspark.sql.functions import when,lit;
emp_df = emp_df.withColumn("is_active", when(col("is_active_flag") == "Yes", lit(True)).otherwise(lit(False)))
emp_df = emp_df.drop("is_active_flag")

In [0]:
spark.sql("create schema if not exists 02_silver_catalog.transformed_schema")
cus_df.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("02_silver_catalog.transformed_schema.customer")
emp_df.write.mode("overwrite").option("overwriteSchema","true").saveAsTable("02_silver_catalog.transformed_schema.employee")